<a href="https://colab.research.google.com/github/jeremy26/hydranets_course/blob/main/Module_2_Depth_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: HydraNet — Depth + Segmentation

In this module, you'll build a **modern HydraNet** that performs **semantic segmentation** and **monocular depth estimation** simultaneously from a single camera image.

**Architecture** (based on [Autoware Vision Pilot](https://github.com/autowarefoundation/autoware_vision_pilot)):

```
Image -> EfficientNet-B0 Backbone -> Context Module -> Shared Neck -> Seg Head
                                                                   -> Depth Head
```

**Key concepts:**
- Shared backbone with multi-scale features
- Context modules for global scene understanding (pseudo-attention)
- U-Net style decoder neck with skip connections
- Task-specific lightweight heads
- Multi-task loss balancing

**Dataset:** BDD100K (driving scenes) + Depth Anything v2 pseudo-depth labels

# 1 — Setup & Installation

In [ ]:
# Clone the course repo and install dependencies
!git clone https://github.com/jeremy26/hydranets_course.git 2>/dev/null || true
%cd hydranets_course

!pip install -q torchvision pillow matplotlib numpy tqdm

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2 — Dataset: BDD100K + Pseudo-Depth

We use [BDD100K](https://www.vis.xyz/bdd100k/), a large-scale driving dataset with:
- **10K images** with pixel-level semantic segmentation (19 classes, Cityscapes-compatible)
- **100K images** with bounding boxes, lane markings, GPS/IMU

BDD100K doesn't include depth annotations, so we generate **pseudo-depth labels** using [Depth Anything v2](https://github.com/DepthAnything/Depth-Anything-V2) — a state-of-the-art monocular depth estimation foundation model. This is a common practice called **knowledge distillation**.

**Downloads:** All data is hosted at [dl.cv.ethz.ch/bdd100k](https://dl.cv.ethz.ch/bdd100k/data/). By downloading, you agree to the [BDD100K license](https://doc.bdd100k.com/download.html).

In [ ]:
# ============================================================
# Download BDD100K 10K images + segmentation labels
# Source: ETH Zurich mirror (https://dl.cv.ethz.ch/bdd100k/)
# ============================================================

DATA_ROOT = "data/bdd100k"
os.makedirs(DATA_ROOT, exist_ok=True)

# 1. Images (10K subset used for segmentation)
!wget -q -O bdd100k_images_10k.zip https://dl.cv.ethz.ch/bdd100k/data/bdd100k_images_10k.zip
!unzip -q bdd100k_images_10k.zip -d data/ && rm bdd100k_images_10k.zip

# 2. Semantic segmentation labels (masks)
!wget -q -O bdd100k_sem_seg.zip https://dl.cv.ethz.ch/bdd100k/data/bdd100k_sem_seg_labels_trainval.zip
!unzip -q bdd100k_sem_seg.zip -d data/ && rm bdd100k_sem_seg.zip

# 3. Generate pseudo-depth labels with Depth Anything v2
# (We provide a script that runs the model on BDD100K images)
# For the workshop, we provide pre-generated depth maps:
!wget -q -O bdd100k_depth_pseudo.zip https://hydranets-data.s3.eu-west-3.amazonaws.com/bdd100k_depth_pseudo.zip
!unzip -q bdd100k_depth_pseudo.zip -d data/bdd100k/labels/ && rm bdd100k_depth_pseudo.zip

# Verify dataset structure
print("Dataset structure:")
for subdir in ['images/10k/train', 'images/10k/val',
               'labels/sem_seg/masks/train', 'labels/sem_seg/masks/val',
               'labels/depth/train', 'labels/depth/val']:
    path = os.path.join(DATA_ROOT, subdir)
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"  {subdir}: {count} files")
    else:
        print(f"  {subdir}: NOT FOUND")

## Visualize the Data

Let's look at a few examples: RGB image, segmentation mask, and pseudo-depth.

In [ ]:
# BDD100K / Cityscapes color palette (19 classes)
BDD_COLORS = np.array([
    [128, 64,128],  # road
    [244, 35,232],  # sidewalk
    [ 70, 70, 70],  # building
    [102,102,156],  # wall
    [190,153,153],  # fence
    [153,153,153],  # pole
    [250,170, 30],  # traffic light
    [220,220,  0],  # traffic sign
    [107,142, 35],  # vegetation
    [152,251,152],  # terrain
    [ 70,130,180],  # sky
    [220, 20, 60],  # person
    [255,  0,  0],  # rider
    [  0,  0,142],  # car
    [  0,  0, 70],  # truck
    [  0, 60,100],  # bus
    [  0, 80,100],  # train
    [  0,  0,230],  # motorcycle
    [119, 11, 32],  # bicycle
], dtype=np.uint8)

BDD_CLASSES = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
               'traffic light', 'traffic sign', 'vegetation', 'terrain',
               'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train',
               'motorcycle', 'bicycle']

def colorize_mask(mask, colors=BDD_COLORS):
    """Convert a class-index mask to an RGB image."""
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id in range(len(colors)):
        rgb[mask == cls_id] = colors[cls_id]
    return rgb

In [ ]:
# Visualize a few samples
img_dir = os.path.join(DATA_ROOT, 'images', '10k', 'train')
if not os.path.exists(img_dir):
    img_dir = os.path.join(DATA_ROOT, 'images', 'train')

seg_dir = os.path.join(DATA_ROOT, 'labels', 'sem_seg', 'masks', 'train')
depth_dir = os.path.join(DATA_ROOT, 'labels', 'depth', 'train')

images = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
for i in range(3):
    idx = np.random.randint(0, len(images))
    basename = os.path.splitext(images[idx])[0]

    # RGB
    img = Image.open(os.path.join(img_dir, images[idx]))
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('RGB Image')
    axes[i, 0].axis('off')

    # Segmentation
    seg_path = os.path.join(seg_dir, basename + '.png')
    if os.path.exists(seg_path):
        seg = np.array(Image.open(seg_path))
        axes[i, 1].imshow(colorize_mask(seg))
    axes[i, 1].set_title('Segmentation')
    axes[i, 1].axis('off')

    # Depth
    depth_path = os.path.join(depth_dir, basename + '.npy')
    if os.path.exists(depth_path):
        depth = np.load(depth_path)
        axes[i, 2].imshow(depth, cmap='magma')
    axes[i, 2].set_title('Pseudo-Depth (Depth Anything v2)')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## DataLoader

We use our custom `BDD100KDataset` class that loads images, segmentation masks, and depth maps. All images are resized to **320x640** (a good balance between quality and Colab speed).

In [ ]:
from utils.data import BDD100KDataset, get_dataloaders, INPUT_SIZE

train_loader, val_loader = get_dataloaders(
    root_dir=DATA_ROOT,
    tasks=['seg', 'depth'],
    batch_size=8,
    num_workers=2
)

print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Input resolution: {INPUT_SIZE}")

# Check a batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Image:  {batch['image'].shape}")
print(f"  Seg:    {batch['seg'].shape}")
print(f"  Depth:  {batch['depth'].shape}")

# 3 — Architecture: Building the HydraNet

Our HydraNet follows the **Autoware Vision Pilot** pattern. Every task goes through 4 stages:

```
Image ─> [Backbone] ─> [Context] ─> [Neck] ─> [Head] ─> Prediction
              │                        ▲
              └── skip connections ─────┘
```

| Component | Role | Trainable? |
|-----------|------|-----------|
| **Backbone** | EfficientNet-B0 — extracts multi-scale features | Pretrained, fine-tuned |
| **Context** | Global Average Pool → MLP → spatial reconstruction → pseudo-attention | Yes |
| **Neck** | U-Net decoder with skip connections from backbone | Yes (shared) |
| **Head** | Lightweight task-specific output layers | Yes (per-task) |

Let's build each component step by step.

## 3.1 — Backbone: EfficientNet-B0

We use **EfficientNet-B0** (pretrained on ImageNet) as our backbone encoder. Unlike the old MobileNetv2, EfficientNet uses compound scaling and is more efficient.

The backbone extracts features at **5 different scales** — these multi-scale features are crucial for the skip connections in the decoder.

In [ ]:
from models.backbone import Backbone

backbone = Backbone(pretrained=True)

# Let's trace the feature dimensions through the backbone
dummy_input = torch.randn(1, 3, 320, 640)
features = backbone(dummy_input)

print("Backbone multi-scale features:")
print(f"{'Level':<10} {'Shape':<30} {'Use'}")
print("-" * 65)
for i, f in enumerate(features):
    use = ['skip to head (H/2)', 'skip to neck block 3 (H/4)',
           'skip to neck block 2 (H/8)', 'skip to neck block 1 (H/16)',
           'deep features -> context (H/32)'][i]
    print(f"feat[{i}]    {str(list(f.shape)):<30} {use}")

## 3.2 — Context Module: Global Scene Understanding

The Context module is what makes this architecture special. Before decoding, we ask: **"What kind of scene am I looking at?"**

It works in 3 steps:
1. **Global Average Pooling** — compress the entire feature map to a single vector
2. **MLP** — process through fully connected layers to capture scene-level patterns
3. **Spatial Reconstruction** — reshape back to a spatial feature map
4. **Pseudo-Attention** — `context * features + features` (like a residual gate)

This helps the network understand global context (e.g., "this is a highway" vs "this is an intersection") before making per-pixel predictions.

In [ ]:
from models.context import SceneContext, DepthContext

seg_context = SceneContext()
depth_context = DepthContext()

# The context module takes the deepest features and returns attention-weighted features
deep_features = features[4]  # (1, 1280, 10, 20)
print(f"Input to context:  {list(deep_features.shape)}")

seg_ctx = seg_context(deep_features)
depth_ctx = depth_context(deep_features)
print(f"Seg context output:   {list(seg_ctx.shape)}")
print(f"Depth context output: {list(depth_ctx.shape)}")
print(f"\nNotice: same shape as input! The context module acts as an attention gate.")

## 3.3 — Neck: U-Net Style Decoder

The Neck upsamples the context-enhanced features back to higher resolution, using **skip connections** from the encoder at each stage.

```
context (1280ch, H/32) ──┐
                          ▼
               [ConvTranspose2d 2x]  +  features[3] (80ch, H/16)  ──> 768ch, H/16
                          ▼
               [ConvTranspose2d 2x]  +  features[2] (40ch, H/8)   ──> 512ch, H/8
                          ▼
               [ConvTranspose2d 2x]  +  features[1] (24ch, H/4)   ──> 256ch, H/4 = NECK OUTPUT
```

The neck is **shared** across tasks — both segmentation and depth use the same decoded features.

In [ ]:
from models.neck import SceneNeck

neck_module = SceneNeck()

neck_output = neck_module(seg_ctx, features)
print(f"Neck output: {list(neck_output.shape)}")
print(f"Resolution: H/4 x W/4 = {320//4} x {640//4}")
print(f"\nThis 256-channel feature map is what all heads receive.")

## 3.4 — Heads: Task-Specific Output Layers

Heads are **lightweight** — they take the shared neck (256ch, H/4) and upsample to full resolution with a few conv layers and one more skip connection from `features[0]`.

- **SegmentationHead** → (B, 19, H, W) — 19-class logits
- **DepthHead** → (B, 1, H, W) — depth prediction

Since heads are small, they train fast. This is key for Module 3 where students will build their own!

In [ ]:
from models.heads import SegmentationHead, DepthHead

seg_head = SegmentationHead(num_classes=19)
depth_head = DepthHead()

seg_out = seg_head(neck_output, features)
depth_out = depth_head(neck_output, features)

print(f"Segmentation output: {list(seg_out.shape)}  (19 class logits at full res)")
print(f"Depth output:        {list(depth_out.shape)}  (1 channel depth at full res)")

## 3.5 — The Full HydraNet

Now let's put it all together. The `HydraNet` class wires: Backbone → Context → Neck → Heads.

In [ ]:
from models.hydranet import HydraNet

model = HydraNet(num_seg_classes=19).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.get_backbone_params())
head_params = sum(p.numel() for p in model.get_head_params())

print(f"Total parameters:    {total_params:,}")
print(f"Backbone parameters: {backbone_params:,} ({100*backbone_params/total_params:.1f}%)")
print(f"Context+Neck+Heads:  {head_params:,} ({100*head_params/total_params:.1f}%)")

# Test forward pass
dummy = torch.randn(2, 3, 320, 640).to(device)
seg_pred, depth_pred = model(dummy)
print(f"\nForward pass OK!")
print(f"Seg prediction:   {list(seg_pred.shape)}")
print(f"Depth prediction: {list(depth_pred.shape)}")

# 4 — Loss Functions

Multi-task learning requires combining multiple losses. We use:

- **Segmentation:** Cross-Entropy Loss (classification per pixel)
- **Depth:** Inverse Huber (berHu) Loss — L1 for small errors, L2 for large errors. More robust than pure L1 or L2.
- **Combined:** Weighted sum with learnable or fixed weights

The **Inverse Huber Loss** is defined as:
- If |error| ≤ c: loss = |error| (like L1)
- If |error| > c: loss = (error² + c²) / 2c (like L2)

where c = 0.2 × max(|error|) per batch.

In [ ]:
from utils.losses import InverseHuberLoss, MultiTaskLoss

# Individual task losses
seg_criterion = nn.CrossEntropyLoss(ignore_index=255)
depth_criterion = InverseHuberLoss()

# Multi-task loss combiner
mtl_loss = MultiTaskLoss(
    task_names=['seg', 'depth'],
    learnable=False,
    initial_weights={'seg': 1.0, 'depth': 1.0}
)

print("Loss functions ready!")

# 5 — Training

We use **differential learning rates**: a lower LR for the pretrained backbone, and a higher LR for the new layers (context, neck, heads). This prevents destroying the pretrained features while allowing the new layers to learn quickly.

In [ ]:
# Optimizer with differential learning rates
optimizer = torch.optim.SGD([
    {'params': model.get_backbone_params(), 'lr': 1e-4},    # Low LR for pretrained backbone
    {'params': model.get_head_params(), 'lr': 1e-2},        # High LR for new layers
], momentum=0.9, weight_decay=1e-5)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[15, 25], gamma=0.1
)

NUM_EPOCHS = 30
print(f"Training for {NUM_EPOCHS} epochs")
print(f"Backbone LR: {optimizer.param_groups[0]['lr']}")
print(f"Heads LR:    {optimizer.param_groups[1]['lr']}")

In [ ]:
from utils.metrics import compute_miou, compute_rmse

def train_one_epoch(model, loader, optimizer, seg_criterion, depth_criterion, mtl_loss, device):
    model.train()
    epoch_losses = {'seg': 0, 'depth': 0, 'total': 0}
    n_batches = 0

    for batch in tqdm(loader, desc='Training', leave=False):
        images = batch['image'].to(device)
        seg_gt = batch['seg'].to(device)
        depth_gt = batch['depth'].to(device)

        # Forward pass
        seg_pred, depth_pred = model(images)

        # Compute individual losses
        loss_seg = seg_criterion(seg_pred, seg_gt)
        loss_depth = depth_criterion(depth_pred, depth_gt)

        # Combine losses
        total_loss, loss_dict = mtl_loss({'seg': loss_seg, 'depth': loss_depth})

        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        for k in epoch_losses:
            epoch_losses[k] += loss_dict[k]
        n_batches += 1

    return {k: v / n_batches for k, v in epoch_losses.items()}


@torch.no_grad()
def validate(model, loader, seg_criterion, depth_criterion, mtl_loss, device, num_classes=19):
    model.eval()
    epoch_losses = {'seg': 0, 'depth': 0, 'total': 0}
    total_miou = 0
    total_rmse = 0
    n_batches = 0

    for batch in tqdm(loader, desc='Validation', leave=False):
        images = batch['image'].to(device)
        seg_gt = batch['seg'].to(device)
        depth_gt = batch['depth'].to(device)

        seg_pred, depth_pred = model(images)

        loss_seg = seg_criterion(seg_pred, seg_gt)
        loss_depth = depth_criterion(depth_pred, depth_gt)
        _, loss_dict = mtl_loss({'seg': loss_seg, 'depth': loss_depth})

        for k in epoch_losses:
            epoch_losses[k] += loss_dict[k]

        # Metrics
        total_miou += compute_miou(seg_pred, seg_gt, num_classes)
        total_rmse += compute_rmse(depth_pred, depth_gt)
        n_batches += 1

    avg_losses = {k: v / n_batches for k, v in epoch_losses.items()}
    avg_losses['miou'] = total_miou / n_batches
    avg_losses['rmse'] = total_rmse / n_batches
    return avg_losses

print("Training functions defined!")

In [ ]:
# Training loop
history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_rmse': []}
best_miou = 0.0

for epoch in range(NUM_EPOCHS):
    # Train
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, seg_criterion, depth_criterion, mtl_loss, device
    )

    # Validate
    val_metrics = validate(
        model, val_loader, seg_criterion, depth_criterion, mtl_loss, device
    )

    scheduler.step()

    # Log
    history['train_loss'].append(train_metrics['total'])
    history['val_loss'].append(val_metrics['total'])
    history['val_miou'].append(val_metrics['miou'])
    history['val_rmse'].append(val_metrics['rmse'])

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_metrics['total']:.4f} | "
          f"Val Loss: {val_metrics['total']:.4f} | "
          f"mIoU: {val_metrics['miou']:.4f} | "
          f"RMSE: {val_metrics['rmse']:.4f}")

    # Save best model
    if val_metrics['miou'] > best_miou:
        best_miou = val_metrics['miou']
        torch.save(model.state_dict(), 'hydranet_best.pth')
        print(f"  -> Saved best model (mIoU: {best_miou:.4f})")

print(f"\nTraining complete! Best mIoU: {best_miou:.4f}")

# 6 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# mIoU
axes[1].plot(history['val_miou'], color='green')
axes[1].set_title('Validation mIoU (Segmentation)')
axes[1].set_xlabel('Epoch')

# RMSE
axes[2].plot(history['val_rmse'], color='orange')
axes[2].set_title('Validation RMSE (Depth)')
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

# 7 — Inference & Visualization

Let's load the best model and visualize predictions on validation images.

In [ ]:
# Load best model
model.load_state_dict(torch.load('hydranet_best.pth', map_location=device))
model.eval()

# Denormalize for visualization
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(img_tensor):
    return (img_tensor.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

# Predict on a few validation images
val_batch = next(iter(val_loader))
images = val_batch['image'].to(device)

with torch.no_grad():
    seg_pred, depth_pred = model(images)

seg_pred = seg_pred.argmax(dim=1).cpu().numpy()
depth_pred = depth_pred.squeeze(1).cpu().numpy()

# Visualize
n_show = min(4, images.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(20, 5 * n_show))

for i in range(n_show):
    # RGB
    axes[i, 0].imshow(denormalize(images[i]))
    axes[i, 0].set_title('Input Image')
    axes[i, 0].axis('off')

    # GT Seg
    axes[i, 1].imshow(colorize_mask(val_batch['seg'][i].numpy()))
    axes[i, 1].set_title('GT Segmentation')
    axes[i, 1].axis('off')

    # Predicted Seg
    axes[i, 2].imshow(colorize_mask(seg_pred[i]))
    axes[i, 2].set_title('Predicted Segmentation')
    axes[i, 2].axis('off')

    # Predicted Depth
    axes[i, 3].imshow(depth_pred[i], cmap='magma')
    axes[i, 3].set_title('Predicted Depth')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

# 8 — FPS Measurement

Let's measure how fast our HydraNet runs — an important metric for autonomous driving!

In [ ]:
import time

model.eval()
dummy = torch.randn(1, 3, 320, 640).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy)

# Benchmark
if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()
n_runs = 100
for _ in range(n_runs):
    with torch.no_grad():
        _ = model(dummy)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start
fps = n_runs / elapsed
print(f"Average inference time: {1000 * elapsed / n_runs:.1f} ms")
print(f"FPS: {fps:.1f}")
print(f"\nNote: Both segmentation AND depth are computed in a single forward pass!")

# 9 — Save Pre-computed Features for Module 3

We save the backbone features for the validation set so students in Module 3 can train new heads **without running the backbone** — making training fast enough for a lab session.

In [ ]:
import os

os.makedirs('precomputed', exist_ok=True)

# Save model weights
torch.save(model.state_dict(), 'precomputed/hydranet_module2.pth')

# Pre-compute and save backbone + neck features for the training set
model.eval()
all_necks = []
all_features_0 = []
all_filenames = []

print("Pre-computing features for Module 3...")
with torch.no_grad():
    for batch in tqdm(train_loader, desc='Pre-computing'):
        images = batch['image'].to(device)

        # Run through backbone and neck
        features = model.backbone(images)
        deep_features = features[4]
        context = model.seg_context(deep_features)
        neck = model.neck(context, features)

        all_necks.append(neck.cpu())
        all_features_0.append(features[0].cpu())
        all_filenames.extend(batch['filename'])

# Save
torch.save({
    'neck': torch.cat(all_necks, dim=0),
    'features_0': torch.cat(all_features_0, dim=0),
    'filenames': all_filenames,
}, 'precomputed/train_features.pt')

print(f"Saved {len(all_filenames)} pre-computed feature sets to precomputed/")
print(f"Neck shape: {torch.cat(all_necks, dim=0).shape}")
print(f"Features[0] shape: {torch.cat(all_features_0, dim=0).shape}")
print(f"\nStudents can now train new heads in minutes!")

# Next Steps

In **Module 3**, you'll learn how to add new heads to this pre-trained HydraNet:
- **Lane Detection** — ego-lane segmentation
- **2D Object Detection** — anchor-free CenterNet-style detection
- **Trajectory Prediction** — steering angle prediction (like Autoware Vision Pilot)

The backbone is frozen, so training new heads takes only **5-15 minutes** on Colab!